###1. Загрузим легкую модель Qwen и на простом промте измерим качество.         
Полученный результат будем считать за baseline, который будем пытаться перебить с помощью   
  a) Улучшения промта (chain of thoughts, детализация и тд).  
  b) Более мощной модели по числу параметров.  
  с) Другой модели.  
  d) Перехода в fine-tuning.  


###2. Специфика данных:   
Эссе оценивается по 4-м критериям, итоговая оценка это просто среднее из них. Поэтому мы хотим от модели попадания в каждый из критериев, помимо общей оценки.

###3. Выбранные метрики качества:  
**MAE** - покажет в баллах на сколько ошибаемся в предсказании, что очень наглядно. Будем в идеале стремиться к тому, чтобы модель ошибалась менее, чем на 0.5 балла  
**RMSE** - учитываем выбросы. Считаем, что дать сильно неправильную оценку за эссе это критически плохо   
**MBE** - модель в реднем завышает или занижает предсказания? Будем стремиться к тому, чтобы модель минимально завышала/занижала оценки.

###4. Формирование пайплайна
Этот же код будем использовать при тесте других моделей/ промтов

In [1]:
# !pip install -q transformers accelerate bitsandbytes pandas datasets torch -- расскомментировать если такой штуки еще не установлено

import torch
print(f"GPU доступен: {torch.cuda.is_available()}")
print(f"Устройство: {torch.cuda.get_device_name(0)}")
print(f"Память: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

GPU доступен: True
Устройство: Tesla T4
Память: 15.6 GB


In [7]:
from google.colab import drive
import pandas as pd

drive.mount('/content/drive')

# Это наш выбранный датасет с каггла, в нем данные сразу разбиты на тест и трейн

!mkdir -p ~/.kaggle
!cp /content/drive/MyDrive/kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d xuanhuynh233/ielts-dataset
!unzip ielts-dataset.zip

csv_path = 'train_final.csv'
df = pd.read_csv(csv_path)

print(df.head())
print(f"Размер датасета: {len(df)} строк.")

# Удалим строки, где 'essay' или 'band' отсутствуют,
# или 'band' не является числом.

initial_len = len(df)
df = df.dropna(subset=['essay', 'band'])
df = df[pd.to_numeric(df['band'], errors='coerce').notna()]
df['band'] = df['band'].astype(float) # Убедимся, что band это float
print(f"\nУдалено {initial_len - len(df)} строк с отсутствующими или некорректными 'essay'/'band'.")
print(f"Осталось {len(df)} строк после очистки.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset URL: https://www.kaggle.com/datasets/xuanhuynh233/ielts-dataset
License(s): MIT
ielts-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)
Archive:  ielts-dataset.zip
replace test_final.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: test_final.csv          
replace train_final.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: train_final.csv         
                                      band  \
0  7.5\n\n\n\n\n\r\r\r\r\r\r\r\r\r\r\r\r\r   
1  5.0\n\n\n\n\n\r\r\r\r\r\r\r\r\r\r\r\r\r   
2  5.5\n\n\n\n\n\r\r\r\r\r\r\r\r\r\r\r\r\r   
3  5.5\n\n\n\n\n\r\r\r\r\r\r\r\r\r\r\r\r\r   
4    4\n\n\n\n\n\r\r\r\r\r\r\r\r\r\r\r\r\r   

                                              prompt  \
0  Interviews form the basic criteria for most la...   
1  Interviews form the basic selecting criteria f...   
2  I

In [8]:
df.head(4)

,band,prompt,essay,TR_Band,TR_Comment,CC_Band,CC_Comment,LR_Band,LR_Mistakes,LR_Corrections,LR_Comment,GRA_Band,GRA_Mistakes,GRA_Corrections,GRA_Comment,Overall_Band,General_Feedback,error
0,7.5,Interviews form the basic criteria for most la...,It is believed by some experts that the tradit...,8.0,The essay sufficiently addresses all parts of ...,8.0,The essay is logically structured and the info...,7.0,exams writing; his teamwork skill is measured;...,written exams; their teamwork skills are measu...,The essay demonstrates a sufficient range of v...,7.0,...his ability to do the work and their proble...,...their ability to do the work and their prob...,"A variety of complex structures is used, and t...",7.5,This is a strong response with a very clear st...,NaN
1,5.0,Interviews form the basic selecting criteria f...,Nowadays numerous huge firms allocate an inter...,5.0,The essay addresses the task only partially. I...,5.0,"The essay is presented with some organization,...",5.0,choosing criteria; up-to-the-minute; reap the ...,selection criterion/criteria; effective/reliab...,The writer attempts to use some less common vo...,5.0,having excellent interpersonal skills are; mak...,having excellent interpersonal skills is; effe...,The essay shows a limited range of sentence st...,5.0,The essay presents a clear position and follow...,NaN
2,5.5,Interview form the basic selection criteria fo...,The interview section is the most vital part o...,6.0,"You have addressed all parts of the question, ...",6.0,The essay has a clear structure with an introd...,5.0,Interview section; prospect for the specified ...,The interview stage; prospective candidate for...,You have used an adequate range of vocabulary ...,5.0,Interview form the basic selection criteria; w...,Interviews form the basic selection criteria; ...,You have attempted to use a mix of simple and ...,5.5,Your essay presents a clear opinion and is log...,NaN
3,5.5,Interviews form the basic selection criteria f...,It is argued that the best method to recruit e...,6.0,The essay addresses the prompt by discussing b...,5.5,"The essay is organized into paragraphs, and th...",5.0,acquired by the recruiter; choosing the best e...,possessed by the applicant; choosing the best ...,The range of vocabulary is limited but minimal...,5.0,who do not lacks in any single question; accor...,who do not lack an answer to any single questi...,The writer attempts to use a mix of simple and...,5.5,Your essay has a clear structure and you prese...,NaN


## загрузка QWEN

In [5]:
#!pip uninstall -y bitsandbytes
#!pip install bitsandbytes>=0.46.1

# Перезапустите сеанс

In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

# Конфигурация 4-bit квантизации - чтобы быстро скачать и чтобы быстро работало (для пробы пера)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model_name = "Qwen/Qwen2.5-3B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Легкая qwen с квантизацией скачивалась 2 минуты - норм

In [2]:
# сразу все импорты
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.stats import spearmanr
import re
from tqdm import tqdm
import torch

In [3]:
# базовый промт
def create_ielts_prompt(essay_to_grade, example_essays):

    prompt = """You are an expert IELTS Writing Task 2 examiner. Grade essays using IELTS band descriptors (0-9, step 0.5).

IELTS CRITERIA (each 0-9):
1. TR (Task Response): How well the essay addresses the task, presents ideas, and supports arguments
2. CC (Coherence & Cohesion): Organization, paragraph structure, and linking devices
3. LR (Lexical Resource): Vocabulary range, accuracy, and appropriateness
4. GRA (Grammatical Range & Accuracy): Grammar variety, accuracy, and complexity

IMPORTANT:
- Use only 0.5 increments (e.g., 6.0, 6.5, 7.0)
- Overall Band = Average of all 4 criteria
- Provide realistic scores based on IELTS standards

Here are graded examples:

"""

    for i, example in enumerate(example_essays, 1):
        prompt += f"""Example {i}:
Essay: "{example['essay']}..."
TR_Band: {example['TR_Band']}
CC_Band: {example['CC_Band']}
LR_Band: {example['LR_Band']}
GRA_Band: {example['GRA_Band']}
Overall_Band: {example['Overall_Band']}
---

"""

    prompt += f"""Now, grade this essay following IELTS standards:

Essay: "{essay_to_grade}"

Provide your evaluation in this EXACT format:
TR_Band: X.X
CC_Band: X.X
LR_Band: X.X
GRA_Band: X.X
Overall_Band: X.X
Explanation: [Brief justification for each criterion]
"""

    return prompt

In [4]:
# получим все оценки из ответа модели, чтобы считать по ним метрики качества

def extract_ielts_scores(response):

    # Извлечем 5 оценок {'TR': float, 'CC': float, 'LR': float, 'GRA': float, 'Overall': float}

    scores = {}

    patterns = {
        'TR': r'TR[_\s]*Band:\s*(\d+(?:\.\d)?)',
        'CC': r'CC[_\s]*Band:\s*(\d+(?:\.\d)?)',
        'LR': r'LR[_\s]*Band:\s*(\d+(?:\.\d)?)',
        'GRA': r'GRA[_\s]*Band:\s*(\d+(?:\.\d)?)',
        'Overall': r'Overall[_\s]*Band:\s*(\d+(?:\.\d)?)'
    }

    for criterion, pattern in patterns.items():
        match = re.search(pattern, response, re.IGNORECASE)
        if match:
            score = float(match.group(1))
            score = round(score * 2) / 2
            scores[criterion] = min(max(score, 0), 9)
        else:
            scores[criterion] = None

    if scores['Overall'] is None and all(scores[k] is not None for k in ['TR', 'CC', 'LR', 'GRA']):
        avg = np.mean([scores['TR'], scores['CC'], scores['LR'], scores['GRA']])
        scores['Overall'] = round(avg * 2) / 2

    return scores

In [5]:
# функция оценки эссе по нашему промту
def grade_ielts_essay(essay, example_essays, model, tokenizer, max_tokens=600):


    prompt = create_ielts_prompt(essay, example_essays)
    # задаем роль
    messages = [
        {"role": "system", "content": "You are an expert IELTS Writing examiner with years of experience."},
        {"role": "user", "content": prompt}
    ]


    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=0.2,  # Очень низкая для стабильности, была 0.3, выдавала разницу в качестве в 1 единцу MAE
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.1
        )

    response = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    )

    return response

In [9]:
# sample_size = 100  # отрабатывает час - больше не хочется повторять этот экспириенс
sample_size = 5
test_df = df.sample(sample_size, random_state=42).copy()

# будем давать 5 примеров оценненых эссе внутри промта
examples = df[~df.index.isin(test_df.index)].sample(5).to_dict('records')

predictions = {
    'TR': [],
    'CC': [],
    'LR': [],
    'GRA': [],
    'Overall': []
}
explanations = []

print(f"Оценка {sample_size} IELTS эссе...")

for idx, row in tqdm(test_df.iterrows(), total=len(test_df)):
    try:
        # Генерация оценки моделью по промту
        response = grade_ielts_essay(
            row['essay'],
            examples,
            model,
            tokenizer,
            max_tokens=600
        )

        # Извлечение оценок из ответа модели
        scores = extract_ielts_scores(response)

        for criterion in ['TR', 'CC', 'LR', 'GRA', 'Overall']:
            predictions[criterion].append(scores[criterion])

        explanations.append(response)

    except Exception as e:
        print(f"\n Ошибка на эссе {idx}: {e}")
        for criterion in predictions.keys():
            predictions[criterion].append(None)
        explanations.append("")

# Добавляем результаты в датафрейм
for criterion in ['TR', 'CC', 'LR', 'GRA', 'Overall']:
    test_df[f'pred_{criterion}'] = predictions[criterion]

test_df['explanation'] = explanations

print("\n Оценка завершена!")

Оценка 5 IELTS эссе...


100%|██████████| 5/5 [05:36<00:00, 67.32s/it]


 Оценка завершена!


#### На каждое эссе тратится около минуты, в будущем хотелось бы ускорить время работы
Тотал - 41 минута на 50 эссе

In [12]:
# оцениваем качество модели

def calculate_ielts_metrics(actual, predicted):

    mask = ~(pd.isna(actual) | pd.isna(predicted))
    actual = np.array(actual)[mask]
    predicted = np.array(predicted)[mask]

    if len(actual) == 0:
        return None

    # MAE
    mae = mean_absolute_error(actual, predicted)

    # RMSE
    rmse = np.sqrt(mean_squared_error(actual, predicted))

    # MBE чтобы понимать занижение/завышение оценок
    mbe = np.mean(predicted - actual)

    # Корреляция Спирмена
    correlation, _ = spearmanr(actual, predicted)

    # Точность
    acc_05 = np.mean(np.abs(predicted - actual) <= 0.5)
    acc_10 = np.mean(np.abs(predicted - actual) <= 1.0)

    # Процент точных совпадений (особенно важно для 0.5 шага)
    exact_match = np.mean(predicted == actual)

    # QWK
    from sklearn.metrics import cohen_kappa_score
    qwk = cohen_kappa_score(
        (actual * 2).astype(int),  # Переводим в целые (умножаем на 2)
        (predicted * 2).astype(int),
        weights='quadratic'
    )

    return {
        'MAE': mae,
        'RMSE': rmse,
        'MBE': mbe,
        'Correlation': correlation,
        'QWK': qwk,
        'Accuracy_±0.5': acc_05,
        'Accuracy_±1.0': acc_10,
        'Exact_Match': exact_match,
        'n_samples': len(actual)
    }


print("метрики качества")

criteria_mapping = {
    'TR': 'TR_Band',
    'CC': 'CC_Band',
    'LR': 'LR_Band',
    'GRA': 'GRA_Band',
    'Overall': 'Overall_Band'
}

all_metrics = {}

for criterion, actual_col in criteria_mapping.items():

    print(f" {criterion} ({actual_col})")
    actual = test_df[actual_col]
    predicted = test_df[f'pred_{criterion}']

    metrics = calculate_ielts_metrics(actual, predicted)

    if metrics:
        all_metrics[criterion] = metrics

        print(f"MAE:              {metrics['MAE']:.3f} bands")
        print(f"RMSE:             {metrics['RMSE']:.3f} bands")
        print(f"MBE:              {metrics['MBE']:+.3f} bands {'(завышение)' if metrics['MBE'] > 0 else '(занижение)' if metrics['MBE'] < 0 else '(нет смещения)'}")
        print(f"Корреляция:       {metrics['Correlation']:.3f}")
        print(f"QWK:              {metrics['QWK']:.3f}")
        print(f"Точность ±0.5:    {metrics['Accuracy_±0.5']:.1%}")
        print(f"Точность ±1.0:    {metrics['Accuracy_±1.0']:.1%}")
        print(f"Точное совпадение: {metrics['Exact_Match']:.1%}")
        print(f"Обработано:       {metrics['n_samples']} эссе")
    else:
        print(" Недостаточно данных для расчета метрик")


метрики качества
 TR (TR_Band)
MAE:              0.800 bands
RMSE:             1.049 bands
MBE:              -0.600 bands (занижение)
Корреляция:       0.354
QWK:              0.143
Точность ±0.5:    60.0%
Точность ±1.0:    80.0%
Точное совпадение: 20.0%
Обработано:       5 эссе
 CC (CC_Band)
MAE:              0.900 bands
RMSE:             1.072 bands
MBE:              -0.500 bands (занижение)
Корреляция:       0.363
QWK:              0.156
Точность ±0.5:    60.0%
Точность ±1.0:    80.0%
Точное совпадение: 0.0%
Обработано:       5 эссе
 LR (LR_Band)
MAE:              1.100 bands
RMSE:             1.245 bands
MBE:              -0.300 bands (занижение)
Корреляция:       0.354
QWK:              0.149
Точность ±0.5:    40.0%
Точность ±1.0:    60.0%
Точное совпадение: 0.0%
Обработано:       5 эссе
 GRA (GRA_Band)
MAE:              0.900 bands
RMSE:             1.072 bands
MBE:              -0.300 bands (занижение)
Корреляция:       0.544
QWK:              0.255
Точность ±0.5:    40.0%
Точно

In [13]:
metrics_df = pd.DataFrame(all_metrics).T
metrics_df = metrics_df.round(3)
print(metrics_df[['MAE', 'RMSE', 'MBE', 'Correlation', 'QWK']])

         MAE   RMSE  MBE  Correlation    QWK
TR       0.8  1.049 -0.6        0.354  0.143
CC       0.9  1.072 -0.5        0.363  0.156
LR       1.1  1.245 -0.3        0.354  0.149
GRA      0.9  1.072 -0.3        0.544  0.255
Overall  0.9  1.118 -0.5        0.354  0.143


## Промежуточные выводы:

- Получили неплохое качество по MAE - по промежуточным оценкам и итоговым отклонение не больше 1,1 балла из 9
- Модель переоценивает максимум на 0,15 балла, в среднем недооценивает на 0,27 балла
- На метрике согласия QWK получили какой-то бред - модель работает практичсеки случайно 🤟

## Конец пайплайна
1) подача промта
2) работа модели на основе этого промта на случайной подвыборке из тестового датасета
3) оценка качества модели



Дальше подробнее посмотрим на вывод модели

In [14]:
# выведем граничные метрики качества по всем оценкам
for criterion in ['TR', 'CC', 'LR', 'GRA', 'Overall']:
    actual = test_df[criteria_mapping[criterion]]
    predicted = test_df[f'pred_{criterion}']

    mask = actual.notna() & predicted.notna()
    errors = (predicted[mask] - actual[mask]).abs()

    print(f"\n{criterion}:")
    print(f"  Макс. ошибка: {errors.max():.1f} bands")
    print(f"  Мин. ошибка:  {errors.min():.1f} bands")
    print(f"  Медиана:      {errors.median():.2f} bands")
    print(f"  95 перцентиль: {errors.quantile(0.95):.2f} bands")


TR:
  Макс. ошибка: 2.0 bands
  Мин. ошибка:  0.0 bands
  Медиана:      0.50 bands
  95 перцентиль: 1.80 bands

CC:
  Макс. ошибка: 2.0 bands
  Мин. ошибка:  0.5 bands
  Медиана:      0.50 bands
  95 перцентиль: 1.80 bands

LR:
  Макс. ошибка: 2.0 bands
  Мин. ошибка:  0.5 bands
  Медиана:      1.00 bands
  95 перцентиль: 1.90 bands

GRA:
  Макс. ошибка: 1.5 bands
  Мин. ошибка:  0.0 bands
  Медиана:      1.00 bands
  95 перцентиль: 1.50 bands

Overall:
  Макс. ошибка: 2.0 bands
  Мин. ошибка:  0.0 bands
  Медиана:      1.00 bands
  95 перцентиль: 1.80 bands


In [15]:
#  вытащим примеры работы модели
valid_overall = test_df.dropna(subset=['pred_Overall'])
valid_overall['error'] = abs(valid_overall['Overall_Band'] - valid_overall['pred_Overall'])


print("\n лучшие варианты")
best = valid_overall.nsmallest(3, 'error')

for idx in best.index:
    row = test_df.loc[idx]

    print(f"Эссе: {row['essay']}...")
    print(f"\nИстинные оценки:")
    print(f"  TR: {row['TR_Band']:.1f}, CC: {row['CC_Band']:.1f}, LR: {row['LR_Band']:.1f}, GRA: {row['GRA_Band']:.1f}")
    print(f"  Overall: {row['Overall_Band']:.1f}")
    print(f"\nПредсказанные оценки:")
    print(f"  TR: {row['pred_TR']:.1f}, CC: {row['pred_CC']:.1f}, LR: {row['pred_LR']:.1f}, GRA: {row['pred_GRA']:.1f}")
    print(f"  Overall: {row['pred_Overall']:.1f}")
    print(f"\n Ошибка Overall: {row['error']:.2f} bands")


print("\n\n худшие варианты")
worst = valid_overall.nlargest(3, 'error')

for idx in worst.index:
    row = test_df.loc[idx]
    print(f"Эссе: {row['essay']}...")
    print(f"\nИстинные оценки:")
    print(f"  TR: {row['TR_Band']:.1f}, CC: {row['CC_Band']:.1f}, LR: {row['LR_Band']:.1f}, GRA: {row['GRA_Band']:.1f}")
    print(f"  Overall: {row['Overall_Band']:.1f}")
    print(f"\nПредсказанные оценки:")
    print(f"  TR: {row['pred_TR']:.1f}, CC: {row['pred_CC']:.1f}, LR: {row['pred_LR']:.1f}, GRA: {row['pred_GRA']:.1f}")
    print(f"  Overall: {row['pred_Overall']:.1f}")
    print(f"\n Ошибка Overall: {row['error']:.2f} bands")


test_df.to_csv('ielts_predictions.csv', index=False)
metrics_df.to_csv('ielts_metrics.csv')



 лучшие варианты
Эссе: It is widely argued that the young are negatively affected by the glamour and riches of celebrities while they are not like that. While I accept that this perception is somewhat justifiable, I believe that not everyone is like that.

On the one hand, there are a number of negative impacts from the effect of celebrities on young people because of several driving factors. First, due to technological development, the influence of stars is also gradually increasing thanks to social networks. Therefore, beware of the "online" glamour or fraudulent wealth of stars. It is a double-edged sword that makes followers affected by that toxicity. For instance, toxic masculinity has been mentioned a lot on social networks in recent times. Andrew Tate, a popular Instagram influencer, inculcated toxic masculinity and spread deviant ideas to his followers, especially guys, through content posted on mobile applications, showing toxic statements, and discriminating against women. I

## Итоги:
Неплохо решаем задачу предсказывания оценки с помощью промптинга, дальше псмотрим получится ли улучшить качество и сможем ли мы как-то оценить еще и фидбек от модели?

# Улучшаем промт

Пока обалденного качества у нас не получилось, попробуем усилить наш промпт.

На данном этапе для нас наиболее важны оценки по критериям, будем просить модель выводить их в начале.

Будем использовать несколько техник промпт инжиниринга, чтобы решить задачу.

План такой:
1. Сделаем Few-Shot контрастным, чтобы науичить модель что есть хорошо, а что плохо
2. Добавим Chain of Thoughts. Кажется, что последовательность в оценке важна
3. Оценка в несколько промптов и усреднение

In [17]:
# функция оценки эссе по любому промпту
def grade_ielts_essay(essay, promt_func, example_essays, model, tokenizer, max_tokens=600):


    prompt = promt_func(essay, example_essays)
    # задаем роль
    messages = [
        {"role": "system", "content": "You are an expert IELTS Writing examiner with years of experience."},
        {"role": "user", "content": prompt}
    ]


    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=0.2,  # Очень низкая для стабильности, была 0.3, выдавала разницу в качестве в 1 единцу MAE
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.1
        )

    response = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    )

    return response

## Контрастный промптинг

Для этого найдем в нашем датасете эссе в наименьшими и наибольшими баллами по каждому критерию и overall score.

In [22]:
def create_ielts_contrast_prompt(essay_to_grade, example_essays):

    prompt = """You are an expert IELTS Writing Task 2 examiner. Grade essays using IELTS band descriptors (0-9, step 0.5).

IELTS CRITERIA (each 0-9):
1. TR (Task Response): How well the essay addresses the task, presents ideas, and supports arguments
2. CC (Coherence & Cohesion): Organization, paragraph structure, and linking devices
3. LR (Lexical Resource): Vocabulary range, accuracy, and appropriateness
4. GRA (Grammatical Range & Accuracy): Grammar variety, accuracy, and complexity

IMPORTANT:
- Use only 0.5 increments (e.g., 6.0, 6.5, 7.0)
- Overall Band = Average of all 4 criteria
- Provide realistic scores based on IELTS standards

Here are graded examples of good and bad scored essays. Pay attention to details of each:

"""

    for i, example in enumerate(example_essays, 1):
        prompt += f"""Example {i}:
Essay: "{example['essay']}..."
TR_Band: {example['TR_Band']}
CC_Band: {example['CC_Band']}
LR_Band: {example['LR_Band']}
GRA_Band: {example['GRA_Band']}
Overall_Band: {example['Overall_Band']}
---

"""

    prompt += f"""Now, grade this essay following IELTS standards:

Essay: "{essay_to_grade}"

Provide your evaluation in this EXACT format in the begginning of your answer:
TR_Band: X.X
CC_Band: X.X
LR_Band: X.X
GRA_Band: X.X
Overall_Band: X.X
Explanation: [Brief justification for each criterion]
"""

    return prompt

In [23]:
metrics = ['TR_Band', 'CC_Band', 'LR_Band', 'GRA_Band', 'Overall_Band']

# Список для хранения индексов отобранных строк
selected_indices = []

# Для каждой метрики ищем минимум и максимум
for metric in metrics:
    for extreme in ['min', 'max']:
        if extreme == 'min':
            # Сортируем по возрастанию, берём первое уникальное эссе
            sorted_df = df.sort_values(by=[metric, 'Overall_Band'], ascending=True)
        else:
            # Для максимума — сортируем по убыванию
            sorted_df = df.sort_values(by=[metric, 'Overall_Band'], ascending=False)

        # Проходим по отсортированным строкам, пока не найдём неиспользованный индекс
        for idx in sorted_df.index:
            if idx not in selected_indices:
                selected_indices.append(idx)
                break

# Теперь создаём новый DataFrame из отобранных строк
contrast_df = df.loc[selected_indices].copy()
contrast_df

,band,prompt,essay,TR_Band,TR_Comment,CC_Band,CC_Comment,LR_Band,LR_Mistakes,LR_Corrections,LR_Comment,GRA_Band,GRA_Mistakes,GRA_Corrections,GRA_Comment,Overall_Band,General_Feedback,error
1645,4.0,It is the responsibility of schools to teach c...,The line graph illustrates the Outokumpu share...,1.0,The response is completely off-topic. The prom...,5.0,The response presents information with some or...,5.0,elevation of the share price; gradually inclin...,increase in the share price; gradually increas...,The writer uses a limited range of vocabulary ...,5.0,from January 2006 and December 2010; there is ...,between January 2006 and December 2010; there ...,A limited range of sentence structures is used...,4.0,The fundamental issue with this response is th...,NaN
136,9.0,The increase in the production of consumer goo...,"These days, there is an ever-growing number of...",9.0,The response fully addresses all parts of the ...,9.0,The essay is exceptionally well-organized. Par...,9.0,product manufacture; in the cable of minimizin...,product manufacturing; capable of minimizing t...,The writer demonstrates masterful control of a...,9.0,Consumer goods are attributed to various facto...,The environmental damage from consumer goods c...,A wide range of complex grammatical structures...,9.0,This is an exemplary response that fulfills al...,NaN
1185,4.0,1.Some people believe that studying at univers...,1. What does an airline clerk say when he/she ...,2.0,The response is entirely unrelated to the prom...,3.0,The text lacks any essay structure. It is pres...,5.0,passengers’s tickets; purpose his/her visit,passengers' tickets; purpose of his/her visit,The vocabulary is limited to the single topic ...,5.0,"he/she want to carry; Now, step I walk / go th...","he/she wants to carry; Now, please step/walk/g...",The response uses a very limited range of gram...,4.0,The most critical issue with this response is ...,NaN
147,9.0,The increase in the production of consumer goo...,"It is a horrendous truth, that rapid growth in...",9.0,The response fully addresses all parts of the ...,9.0,"The essay is skillfully managed, with a clear ...",9.0,product manufacturing causing the issues; good...,the manufacturing of products causes these iss...,The writer demonstrates a wide vocabulary and ...,9.0,"It is a horrendous truth, that rapid growth......",It is a horrendous truth that rapid growth...;...,A wide range of grammatical structures is used...,9.0,This is a very strong response that successful...,NaN
4,4.0,Interviews from the basic selecting criteria f...,Nowadays many companies conduct interviews bef...,4.0,The response addresses the task only in a mini...,4.0,"The essay presents information and ideas, but ...",4.0,chosing; intetviuvs; campaigns; sending them; ...,choosing; interviews; companies; hiring them; ...,The writer uses only basic and repetitive voca...,4.0,Interviews from the basic selecting criteria; ...,Interviews form the basic selection criteria; ...,Only a very limited range of sentence structur...,4.0,"Your essay attempts to address the prompt, but...",NaN
395,9.0,TASK 2: Some people think the money spent on d...,Some individuals claim that the amount of mone...,9.0,The response fully and expertly addresses all ...,9.0,The essay demonstrates masterful control of co...,9.0,world folk; human beings' specimen; the new cr...,the world's population / people worldwide; the...,"The use of vocabulary is wide-ranging, sophist...",9.0,is public funds consuming that could be utilis...,consumes public funds that could be utilised; ...,The essay showcases a wide range of complex gr...,9.0,This is an outstanding essay that demonstrates...,NaN
6119,4.0,Nowadays more and more jobs and tasks are done...,There are many companies and factories to comp...,5.0,The response attempts to address the prompt by...,4.0,The essay shows a basic attempt at paragraphin...,4.0,"to complete manual task, which is traditionall...","to complete manual tasks, which were 

In [26]:
# sample_size = 100  # отрабатывает час - больше не хочется повторять этот экспириенс
sample_size = 50
test_df = df.sample(sample_size, random_state=42).copy()

examples = contrast_df.to_dict('records')

predictions = {
    'TR': [],
    'CC': [],
    'LR': [],
    'GRA': [],
    'Overall': []
}
explanations = []

print(f"Оценка {sample_size} IELTS эссе...")

for idx, row in tqdm(test_df.iterrows(), total=len(test_df)):
    try:
        # Генерация оценки моделью по промту
        response = grade_ielts_essay(
            row['essay'],
            create_ielts_contrast_prompt,
            examples,
            model,
            tokenizer,
            max_tokens=600
        )

        # Извлечение оценок из ответа модели
        scores = extract_ielts_scores(response)

        for criterion in ['TR', 'CC', 'LR', 'GRA', 'Overall']:
            predictions[criterion].append(scores[criterion])

        explanations.append(response)

    except Exception as e:
        print(f"\n Ошибка на эссе {idx}: {e}")
        for criterion in predictions.keys():
            predictions[criterion].append(None)
        explanations.append("")

# Добавляем результаты в датафрейм
for criterion in ['TR', 'CC', 'LR', 'GRA', 'Overall']:
    test_df[f'pred_{criterion}'] = predictions[criterion]

test_df['explanation'] = explanations

print("\n Оценка завершена!")

Оценка 50 IELTS эссе...


100%|██████████| 50/50 [45:32<00:00, 54.65s/it]


 Оценка завершена!


In [27]:
print("метрики качества")

criteria_mapping = {
    'TR': 'TR_Band',
    'CC': 'CC_Band',
    'LR': 'LR_Band',
    'GRA': 'GRA_Band',
    'Overall': 'Overall_Band'
}

all_metrics = {}

for criterion, actual_col in criteria_mapping.items():

    print(f" {criterion} ({actual_col})")
    actual = test_df[actual_col]
    predicted = test_df[f'pred_{criterion}']

    metrics = calculate_ielts_metrics(actual, predicted)

    if metrics:
        all_metrics[criterion] = metrics

        print(f"MAE:              {metrics['MAE']:.3f} bands")
        print(f"RMSE:             {metrics['RMSE']:.3f} bands")
        print(f"MBE:              {metrics['MBE']:+.3f} bands {'(завышение)' if metrics['MBE'] > 0 else '(занижение)' if metrics['MBE'] < 0 else '(нет смещения)'}")
        print(f"Корреляция:       {metrics['Correlation']:.3f}")
        print(f"QWK:              {metrics['QWK']:.3f}")
        print(f"Точность ±0.5:    {metrics['Accuracy_±0.5']:.1%}")
        print(f"Точность ±1.0:    {metrics['Accuracy_±1.0']:.1%}")
        print(f"Точное совпадение: {metrics['Exact_Match']:.1%}")
        print(f"Обработано:       {metrics['n_samples']} эссе")
    else:
        print(" Недостаточно данных для расчета метрик")


метрики качества
 TR (TR_Band)
MAE:              1.280 bands
RMSE:             1.670 bands
MBE:              +0.700 bands (завышение)
Корреляция:       0.158
QWK:              0.103
Точность ±0.5:    28.0%
Точность ±1.0:    66.0%
Точное совпадение: 14.0%
Обработано:       50 эссе
 CC (CC_Band)
MAE:              1.220 bands
RMSE:             1.587 bands
MBE:              +0.460 bands (завышение)
Корреляция:       0.269
QWK:              0.246
Точность ±0.5:    28.0%
Точность ±1.0:    70.0%
Точное совпадение: 14.0%
Обработано:       50 эссе
 LR (LR_Band)
MAE:              1.360 bands
RMSE:             1.766 bands
MBE:              +0.880 bands (завышение)
Корреляция:       0.243
QWK:              0.177
Точность ±0.5:    28.0%
Точность ±1.0:    60.0%
Точное совпадение: 16.0%
Обработано:       50 эссе
 GRA (GRA_Band)
MAE:              1.340 bands
RMSE:             1.735 bands
MBE:              +0.880 bands (завышение)
Корреляция:       0.271
QWK:              0.161
Точность ±0.5:    30.0%


In [28]:
metrics_df = pd.DataFrame(all_metrics).T
metrics_df = metrics_df.round(3)
print(metrics_df[['MAE', 'RMSE', 'MBE', 'Correlation', 'QWK']])

          MAE   RMSE   MBE  Correlation    QWK
TR       1.28  1.670  0.70        0.158  0.103
CC       1.22  1.587  0.46        0.269  0.246
LR       1.36  1.766  0.88        0.243  0.177
GRA      1.34  1.735  0.88        0.271  0.161
Overall  1.29  1.663  0.67        0.215  0.140


По сравнению с базовым промптом

Несмотря на большую точность, базовый промт хуже различает работы разного качества (низкая корреляция и QWK). Контрастный промт, хотя и менее точен, лучше ранжирует эссе (QWK выше в ~8 раз)

## Chain of thoughts
Добавим цепочку мыслей и самопроверку

In [29]:
def create_ielts_contrast_prompt_COT(essay_to_grade, example_essays):

    prompt = """You are an expert IELTS Writing Task 2 examiner. Grade essays using IELTS band descriptors (0-9, step 0.5).

IELTS CRITERIA (each 0-9):
1. TR (Task Response): How well the essay addresses the task, presents ideas, and supports arguments
2. CC (Coherence & Cohesion): Organization, paragraph structure, and linking devices
3. LR (Lexical Resource): Vocabulary range, accuracy, and appropriateness
4. GRA (Grammatical Range & Accuracy): Grammar variety, accuracy, and complexity

IMPORTANT:
- Use only 0.5 increments (e.g., 6.0, 6.5, 7.0)
- Overall Band = Average of all 4 criteria
- Provide realistic scores based on IELTS standards

Here are graded examples of good and bad scored essays. Pay attention to details of each:

"""

    for i, example in enumerate(example_essays, 1):
        prompt += f"""Example {i}:
Essay: "{example['essay']}..."
TR_Band: {example['TR_Band']}
CC_Band: {example['CC_Band']}
LR_Band: {example['LR_Band']}
GRA_Band: {example['GRA_Band']}
Overall_Band: {example['Overall_Band']}
---

"""

    prompt += f"""
After scoring, perform self-verification: Would a second examiner agree? If not, revise

Now, grade this essay following IELTS standards:

Essay: "{essay_to_grade}"

--- INTERNAL REASONING (DO NOT OUTPUT THIS) ---
Think step by step. For each criterion, compare to provided examples.
Then self-verify: Would a second examiner agree? If not, revise your scores.

--- FINAL OUTPUT (ONLY THIS SECTION) Provide your evaluation in this EXACT format in the begginning of your answer: ---
TR_Band: X.X
CC_Band: X.X
LR_Band: X.X
GRA_Band: X.X
Overall_Band: X.X
Explanation: [Brief justification, max 2 sentences per criterion]
"""

    return prompt

In [32]:
sample_size = 50
test_df = df.sample(sample_size, random_state=42).copy()

examples = contrast_df.to_dict('records')

predictions = {
    'TR': [],
    'CC': [],
    'LR': [],
    'GRA': [],
    'Overall': []
}
explanations = []

print(f"Оценка {sample_size} IELTS эссе...")

for idx, row in tqdm(test_df.iterrows(), total=len(test_df)):
    try:
        # Генерация оценки моделью по промту
        response = grade_ielts_essay(
            row['essay'],
            create_ielts_contrast_prompt_COT,
            examples,
            model,
            tokenizer,
            max_tokens=600
        )

        # Извлечение оценок из ответа модели
        scores = extract_ielts_scores(response)

        for criterion in ['TR', 'CC', 'LR', 'GRA', 'Overall']:
            predictions[criterion].append(scores[criterion])

        explanations.append(response)

    except Exception as e:
        print(f"\n Ошибка на эссе {idx}: {e}")
        for criterion in predictions.keys():
            predictions[criterion].append(None)
        explanations.append("")

# Добавляем результаты в датафрейм
for criterion in ['TR', 'CC', 'LR', 'GRA', 'Overall']:
    test_df[f'pred_{criterion}'] = predictions[criterion]

test_df['explanation'] = explanations

print("\n Оценка завершена!")

Оценка 50 IELTS эссе...


100%|██████████| 50/50 [35:42<00:00, 42.86s/it]


 Оценка завершена!


In [33]:
print("метрики качества")

criteria_mapping = {
    'TR': 'TR_Band',
    'CC': 'CC_Band',
    'LR': 'LR_Band',
    'GRA': 'GRA_Band',
    'Overall': 'Overall_Band'
}

all_metrics = {}

for criterion, actual_col in criteria_mapping.items():

    print(f" {criterion} ({actual_col})")
    actual = test_df[actual_col]
    predicted = test_df[f'pred_{criterion}']

    metrics = calculate_ielts_metrics(actual, predicted)

    if metrics:
        all_metrics[criterion] = metrics

        print(f"MAE:              {metrics['MAE']:.3f} bands")
        print(f"RMSE:             {metrics['RMSE']:.3f} bands")
        print(f"MBE:              {metrics['MBE']:+.3f} bands {'(завышение)' if metrics['MBE'] > 0 else '(занижение)' if metrics['MBE'] < 0 else '(нет смещения)'}")
        print(f"Корреляция:       {metrics['Correlation']:.3f}")
        print(f"QWK:              {metrics['QWK']:.3f}")
        print(f"Точность ±0.5:    {metrics['Accuracy_±0.5']:.1%}")
        print(f"Точность ±1.0:    {metrics['Accuracy_±1.0']:.1%}")
        print(f"Точное совпадение: {metrics['Exact_Match']:.1%}")
        print(f"Обработано:       {metrics['n_samples']} эссе")
    else:
        print(" Недостаточно данных для расчета метрик")


метрики качества
 TR (TR_Band)
MAE:              1.402 bands
RMSE:             1.852 bands
MBE:              +0.641 bands (завышение)
Корреляция:       -0.035
QWK:              -0.076
Точность ±0.5:    30.4%
Точность ±1.0:    56.5%
Точное совпадение: 17.4%
Обработано:       46 эссе
 CC (CC_Band)
MAE:              1.380 bands
RMSE:             1.822 bands
MBE:              +0.402 bands (завышение)
Корреляция:       0.038
QWK:              -0.035
Точность ±0.5:    28.3%
Точность ±1.0:    63.0%
Точное совпадение: 17.4%
Обработано:       46 эссе
 LR (LR_Band)
MAE:              1.587 bands
RMSE:             1.981 bands
MBE:              +0.848 bands (завышение)
Корреляция:       0.008
QWK:              -0.055
Точность ±0.5:    19.6%
Точность ±1.0:    52.2%
Точное совпадение: 8.7%
Обработано:       46 эссе
 GRA (GRA_Band)
MAE:              1.565 bands
RMSE:             1.950 bands
MBE:              +0.848 bands (завышение)
Корреляция:       0.002
QWK:              -0.040
Точность ±0.5:    21

In [34]:
metrics_df = pd.DataFrame(all_metrics).T
metrics_df = metrics_df.round(3)
print(metrics_df[['MAE', 'RMSE', 'MBE', 'Correlation', 'QWK']])

           MAE   RMSE    MBE  Correlation    QWK
TR       1.402  1.852  0.641       -0.035 -0.076
CC       1.380  1.822  0.402        0.038 -0.035
LR       1.587  1.981  0.848        0.008 -0.055
GRA      1.565  1.950  0.848        0.002 -0.040
Overall  1.446  1.878  0.620       -0.018 -0.071


Все плохо((

Попробуем доработать - выводить явные рассуждения в ответе и увеличить количество выходных токенов


## COT 2

In [35]:
def create_ielts_contrast_prompt_COT_v2(essay_to_grade, example_essays):
    prompt = """You are an expert IELTS Writing Task 2 examiner. Grade essays using IELTS band descriptors (0-9, step 0.5).

IELTS CRITERIA (each 0-9):
1. TR (Task Response): How well the essay addresses the task, presents ideas, and supports arguments
2. CC (Coherence & Cohesion): Organization, paragraph structure, and linking devices
3. LR (Lexical Resource): Vocabulary range, accuracy, and appropriateness
4. GRA (Grammatical Range & Accuracy): Grammar variety, accuracy, and complexity

RULES:
- Use only 0.5 increments (e.g., 6.0, 6.5, 7.0)
- Overall Band = Average of all 4 criteria (round to nearest 0.5)
- Be realistic: most IELTS essays score between 5.0 and 8.0

REFERENCE EXAMPLES (pay attention to patterns):

"""
    # Добавляем только 4-5 лучших примера (не все)
    for i, example in enumerate(example_essays[:5], 1):
        prompt += f"""
Example {i}:
Essay excerpt: "{example['essay'][:200]}..."
TR: {example['TR_Band']} | CC: {example['CC_Band']} | LR: {example['LR_Band']} | GRA: {example['GRA_Band']} | Overall: {example['Overall_Band']}
"""

    prompt += f"""
Now grade the following essay:

ESSAY TO GRADE:
\"\"\"{essay_to_grade}\"\"\"

---
CHAIN-OF-THOUGHT REASONING (write your analysis below):

1. TR (Task Response) analysis:
   - Does the essay fully address all parts of the task?
   - Are main ideas supported with examples/reasons?
   - [Your 1-2 sentence analysis]
   → TR score: [X.X]

2. CC (Coherence & Cohesion) analysis:
   - Is there logical paragraph organization?
   - Are linking devices used effectively?
   - [Your 1-2 sentence analysis]
   → CC score: [X.X]

3. LR (Lexical Resource) analysis:
   - Is vocabulary range sufficient for the task?
   - Are word choices precise and natural?
   - [Your 1-2 sentence analysis]
   → LR score: [X.X]

4. GRA (Grammatical Range & Accuracy) analysis:
   - Are sentence structures varied?
   - Are there grammatical errors that impede understanding?
   - [Your 1-2 sentence analysis]
   → GRA score: [X.X]

5. Self-verification:
   - Would a second examiner agree with these scores?
   - Are scores consistent with the reference examples?
   - If not, which score(s) would I adjust?

---
FINAL ANSWER (use this exact format at the end of your response):

TR_Band: X.X
CC_Band: X.X
LR_Band: X.X
GRA_Band: X.X
Overall_Band: X.X

Explanation: [1 sentence summary]
"""

    return prompt



In [37]:
sample_size = 50
test_df = df.sample(sample_size, random_state=42).copy()

examples = contrast_df[:5].to_dict('records')

predictions = {
    'TR': [],
    'CC': [],
    'LR': [],
    'GRA': [],
    'Overall': []
}
explanations = []

print(f"Оценка {sample_size} IELTS эссе...")

for idx, row in tqdm(test_df.iterrows(), total=len(test_df)):
    try:
        # Генерация оценки моделью по промту
        response = grade_ielts_essay(
            row['essay'],
            create_ielts_contrast_prompt_COT_v2,
            examples,
            model,
            tokenizer,
            max_tokens=2000
        )

        # Извлечение оценок из ответа модели
        scores = extract_ielts_scores(response)

        for criterion in ['TR', 'CC', 'LR', 'GRA', 'Overall']:
            predictions[criterion].append(scores[criterion])

        explanations.append(response)

    except Exception as e:
        print(f"\n Ошибка на эссе {idx}: {e}")
        for criterion in predictions.keys():
            predictions[criterion].append(None)
        explanations.append("")

# Добавляем результаты в датафрейм
for criterion in ['TR', 'CC', 'LR', 'GRA', 'Overall']:
    test_df[f'pred_{criterion}'] = predictions[criterion]

test_df['explanation'] = explanations

print("\n Оценка завершена!")

Оценка 50 IELTS эссе...


100%|██████████| 50/50 [46:30<00:00, 55.81s/it]


 Оценка завершена!


In [38]:
print("метрики качества")

criteria_mapping = {
    'TR': 'TR_Band',
    'CC': 'CC_Band',
    'LR': 'LR_Band',
    'GRA': 'GRA_Band',
    'Overall': 'Overall_Band'
}

all_metrics = {}

for criterion, actual_col in criteria_mapping.items():

    print(f" {criterion} ({actual_col})")
    actual = test_df[actual_col]
    predicted = test_df[f'pred_{criterion}']

    metrics = calculate_ielts_metrics(actual, predicted)

    if metrics:
        all_metrics[criterion] = metrics

        print(f"MAE:              {metrics['MAE']:.3f} bands")
        print(f"RMSE:             {metrics['RMSE']:.3f} bands")
        print(f"MBE:              {metrics['MBE']:+.3f} bands {'(завышение)' if metrics['MBE'] > 0 else '(занижение)' if metrics['MBE'] < 0 else '(нет смещения)'}")
        print(f"Корреляция:       {metrics['Correlation']:.3f}")
        print(f"QWK:              {metrics['QWK']:.3f}")
        print(f"Точность ±0.5:    {metrics['Accuracy_±0.5']:.1%}")
        print(f"Точность ±1.0:    {metrics['Accuracy_±1.0']:.1%}")
        print(f"Точное совпадение: {metrics['Exact_Match']:.1%}")
        print(f"Обработано:       {metrics['n_samples']} эссе")
    else:
        print(" Недостаточно данных для расчета метрик")


метрики качества
 TR (TR_Band)
MAE:              1.173 bands
RMSE:             1.437 bands
MBE:              +0.724 bands (завышение)
Корреляция:       0.148
QWK:              0.042
Точность ±0.5:    32.7%
Точность ±1.0:    57.1%
Точное совпадение: 14.3%
Обработано:       49 эссе
 CC (CC_Band)
MAE:              1.173 bands
RMSE:             1.405 bands
MBE:              +0.480 bands (завышение)
Корреляция:       0.207
QWK:              0.094
Точность ±0.5:    28.6%
Точность ±1.0:    59.2%
Точное совпадение: 12.2%
Обработано:       49 эссе
 LR (LR_Band)
MAE:              1.224 bands
RMSE:             1.467 bands
MBE:              +0.571 bands (завышение)
Корреляция:       0.088
QWK:              0.025
Точность ±0.5:    32.7%
Точность ±1.0:    53.1%
Точное совпадение: 12.2%
Обработано:       49 эссе
 GRA (GRA_Band)
MAE:              1.204 bands
RMSE:             1.414 bands
MBE:              +0.531 bands (завышение)
Корреляция:       0.036
QWK:              -0.003
Точность ±0.5:    36.7%

In [39]:
metrics_df = pd.DataFrame(all_metrics).T
metrics_df = metrics_df.round(3)
print(metrics_df[['MAE', 'RMSE', 'MBE', 'Correlation', 'QWK']])

           MAE   RMSE    MBE  Correlation    QWK
TR       1.173  1.437  0.724        0.148  0.042
CC       1.173  1.405  0.480        0.207  0.094
LR       1.224  1.467  0.571        0.088  0.025
GRA      1.204  1.414  0.531        0.036 -0.003
Overall  1.194  1.387  0.459       -0.064 -0.027


Лучше, но не так хорошо, как базовый и контрастный

Хочется попробовать запустить контрастный промп с экспертными подсказками, так как он показал лучший результат

## Экспертные комментарии

In [40]:
def create_ielts_contrast_prompt_enhanced(essay_to_grade, example_essays):

    prompt = """You are an expert IELTS Writing Task 2 examiner with 10+ years of experience. Grade essays using official IELTS band descriptors.

IELTS CRITERIA WITH EXPERT GUIDELINES (0-9, step 0.5)

1. TR (Task Response):
   5.0 → Addresses task only partially, limited development
   6.0 → Addresses task, some development but may lack focus
   7.0 → Addresses all parts, clear position, well-developed ideas
   8.0 → Fully addresses all parts, sustained position, relevant ideas

   EXPERT TIP: Check if EVERY part of the question is answered

2. CC (Coherence & Cohesion):
   5.0 → Some organization but not logical, limited linking
   6.0 → Clear overall progression, adequate paragraphing
   7.0 → Logically organizes information, clear progression
   8.0 → Manages information skillfully, seamless cohesion

   EXPERT TIP: Read first sentence of each paragraph = essay outline

3. LR (Lexical Resource):
   5.0 → Limited vocabulary, noticeable errors
   6.0 → Adequate range, some errors in word choice
   7.0 → Good range, some flexibility and precision
   8.0 → Wide range, natural and sophisticated control

   EXPERT TIP: Look for topic-specific vocabulary, not just "good words"

4. GRA (Grammatical Range & Accuracy):
   5.0 → Limited structures, frequent errors
   6.0 → Mix of simple/complex, some errors
   7.0 → Variety of structures, majority error-free
   8.0 → Wide variety, most sentences error-free

   EXPERT TIP: Count errors per 100 words (>8 errors = 5.0)


REFERENCE EXAMPLES (Pay attention to patterns)
"""

    # Добавляем примеры с аннотациями сильных/слабых сторон
    for i, example in enumerate(example_essays[:6], 1):
        prompt += f"""
EXAMPLE {i}:
Essay excerpt: "{example['essay']}"
│ TR: {example['TR_Band']}  │ CC: {example['CC_Band']}  │ LR: {example['LR_Band']}  │ GRA: {example['GRA_Band']}  │ Overall: {example['Overall_Band']} │

"""

    prompt += f"""

ESSAY TO GRADE

\"\"\"{essay_to_grade}\"\"\"

GRADING PROCESS (Follow these steps)

STEP 1 - Quick Scan (30 sec):
- Does essay answer ALL parts of the question? [YES/NO]
- Is there clear paragraph structure? [YES/NO]
- Estimated band range? [5.0-6.0 / 6.0-7.0 / 7.0-8.0]

STEP 2 - Detailed Analysis per Criterion:

TR:
- Task completion: ___%
- Position clear? [YES/NO/ PARTLY]
- Ideas developed? [POOR / ADEQUATE / GOOD / EXCELLENT]
→ Score: ___

CC:
- Paragraphs logical? [YES/NO/ PARTLY]
- Linking devices used? [FEW / SOME / MANY]
- Progression clear? [NO / SOMETIMES / YES]
→ Score: ___

LR:
- Vocabulary range: [BASIC / ADEQUATE / GOOD / WIDE]
- Word choice precision: [POOR / OKAY / GOOD / NATURAL]
- Errors noticeable? [MANY / SOME / FEW / NONE]
→ Score: ___

GRA:
- Sentence variety: [SIMPLE ONLY / SOME COMPLEX / GOOD VARIETY]
- Error density (per 100 words): [>10 / 6-10 / 3-5 / <3]
- Errors impede understanding? [YES / SOMETIMES / NO]
→ Score: ___

STEP 3 - Cross-check with Reference Examples:
- Which example is most similar in quality? Example #___
- Are my scores consistent with that example? [YES/NO]
- If NO, adjust scores DOWN by 0.5

STEP 4 - Final Calibration:
- IMPORTANT: IELTS examiners RARELY give scores above 8.0
- Most essays score between 5.0 and 7.5
- If your average > 7.5, review if it's truly exceptional

COMMON MISTAKES TO AVOID (Based on calibration data)

DO NOT over-score: Average essay = 6.0-6.5, NOT 7.0
DO NOT give 8.0+ unless truly exceptional (top 5%)
DO NOT ignore small errors - they matter
DO NOT give same score for all criteria without justification

DO compare to reference examples
DO check if overall band matches typical IELTS distribution
DO adjust DOWN if unsure (strict grading is more accurate)

RESPONSE FORMAT

TR_Band: X.X
CC_Band: X.X
LR_Band: X.X
GRA_Band: X.X
Overall_Band: X.X

Explanation:
TR: [1 sentence]
CC: [1 sentence]
LR: [1 sentence]
GRA: [1 sentence]

Calibration check: [Confirm you avoided common mistakes]
"""

    return prompt

In [44]:
sample_size = 50
test_df = df.sample(sample_size, random_state=42).copy()

examples = contrast_df[:4].to_dict('records')

predictions = {
    'TR': [],
    'CC': [],
    'LR': [],
    'GRA': [],
    'Overall': []
}
explanations = []

print(f"Оценка {sample_size} IELTS эссе...")

for idx, row in tqdm(test_df.iterrows(), total=len(test_df)):
    try:
        # Генерация оценки моделью по промту
        response = grade_ielts_essay(
            row['essay'],
            create_ielts_contrast_prompt_enhanced,
            examples,
            model,
            tokenizer,
            max_tokens=2000
        )

        # Извлечение оценок из ответа модели
        scores = extract_ielts_scores(response)

        for criterion in ['TR', 'CC', 'LR', 'GRA', 'Overall']:
            predictions[criterion].append(scores[criterion])

        explanations.append(response)

    except Exception as e:
        print(f"\n Ошибка на эссе {idx}: {e}")
        for criterion in predictions.keys():
            predictions[criterion].append(None)
        explanations.append("")

# Добавляем результаты в датафрейм
for criterion in ['TR', 'CC', 'LR', 'GRA', 'Overall']:
    test_df[f'pred_{criterion}'] = predictions[criterion]

test_df['explanation'] = explanations

print("\n Оценка завершена!")

Оценка 50 IELTS эссе...


100%|██████████| 50/50 [1:32:04<00:00, 110.48s/it]


 Оценка завершена!


In [45]:
print("метрики качества")

criteria_mapping = {
    'TR': 'TR_Band',
    'CC': 'CC_Band',
    'LR': 'LR_Band',
    'GRA': 'GRA_Band',
    'Overall': 'Overall_Band'
}

all_metrics = {}

for criterion, actual_col in criteria_mapping.items():

    print(f" {criterion} ({actual_col})")
    actual = test_df[actual_col]
    predicted = test_df[f'pred_{criterion}']

    metrics = calculate_ielts_metrics(actual, predicted)

    if metrics:
        all_metrics[criterion] = metrics

        print(f"MAE:              {metrics['MAE']:.3f} bands")
        print(f"RMSE:             {metrics['RMSE']:.3f} bands")
        print(f"MBE:              {metrics['MBE']:+.3f} bands {'(завышение)' if metrics['MBE'] > 0 else '(занижение)' if metrics['MBE'] < 0 else '(нет смещения)'}")
        print(f"Корреляция:       {metrics['Correlation']:.3f}")
        print(f"QWK:              {metrics['QWK']:.3f}")
        print(f"Точность ±0.5:    {metrics['Accuracy_±0.5']:.1%}")
        print(f"Точность ±1.0:    {metrics['Accuracy_±1.0']:.1%}")
        print(f"Точное совпадение: {metrics['Exact_Match']:.1%}")
        print(f"Обработано:       {metrics['n_samples']} эссе")
    else:
        print(" Недостаточно данных для расчета метрик")


метрики качества
 TR (TR_Band)
MAE:              1.391 bands
RMSE:             1.830 bands
MBE:              +0.565 bands (завышение)
Корреляция:       -0.134
QWK:              -0.028
Точность ±0.5:    30.4%
Точность ±1.0:    65.2%
Точное совпадение: 13.0%
Обработано:       23 эссе
 CC (CC_Band)
MAE:              1.543 bands
RMSE:             1.920 bands
MBE:              +0.457 bands (завышение)
Корреляция:       -0.187
QWK:              -0.187
Точность ±0.5:    26.1%
Точность ±1.0:    56.5%
Точное совпадение: 13.0%
Обработано:       23 эссе
 LR (LR_Band)
MAE:              1.587 bands
RMSE:             1.964 bands
MBE:              +1.109 bands (завышение)
Корреляция:       -0.101
QWK:              -0.067
Точность ±0.5:    21.7%
Точность ±1.0:    52.2%
Точное совпадение: 17.4%
Обработано:       23 эссе
 GRA (GRA_Band)
MAE:              1.522 bands
RMSE:             1.989 bands
MBE:              +1.130 bands (завышение)
Корреляция:       -0.094
QWK:              -0.066
Точность ±0.5:  

In [46]:
metrics_df = pd.DataFrame(all_metrics).T
metrics_df = metrics_df.round(3)
print(metrics_df[['MAE', 'RMSE', 'MBE', 'Correlation', 'QWK']])

           MAE   RMSE    MBE  Correlation    QWK
TR       1.391  1.830  0.565       -0.134 -0.028
CC       1.543  1.920  0.457       -0.187 -0.187
LR       1.587  1.964  1.109       -0.101 -0.067
GRA      1.522  1.989  1.130       -0.094 -0.066
Overall  1.457  1.766  0.848       -0.037 -0.043


Пока кажется, что чем сложнее промт, тем ниже результаты.

Хочется оставить только контрастность и попробовать добавить в промпт комментарии экспертов по эссе, может это улучшит качество


## Добавление примеров комментариев в промпт

In [48]:
def create_ielts_contrast_prompt_with_ans(essay_to_grade, example_essays):

    prompt = """You are an expert IELTS Writing Task 2 examiner. Grade essays using IELTS band descriptors (0-9, step 0.5).

IELTS CRITERIA (each 0-9):
1. TR (Task Response): How well the essay addresses the task, presents ideas, and supports arguments
2. CC (Coherence & Cohesion): Organization, paragraph structure, and linking devices
3. LR (Lexical Resource): Vocabulary range, accuracy, and appropriateness
4. GRA (Grammatical Range & Accuracy): Grammar variety, accuracy, and complexity

IMPORTANT:
- Use only 0.5 increments (e.g., 6.0, 6.5, 7.0)
- Overall Band = Average of all 4 criteria
- Provide realistic scores based on IELTS standards

Here are graded examples of good and bad scored essays. Pay attention to details of each:

"""

    for i, example in enumerate(example_essays, 1):
        prompt += f"""Example {i}:
Essay: "{example['essay']}"
TR_Band: {example['TR_Band']}
CC_Band: {example['CC_Band']}
LR_Band: {example['LR_Band']}
GRA_Band: {example['GRA_Band']}
Overall_Band: {example['Overall_Band']}
TR_Comment: {example['TR_Comment']}
GRA_Comment: {example['GRA_Comment']}
LR_Comment: {example['LR_Comment']}
CC_Comment: {example['CC_Comment']}
---

"""

    prompt += f"""Now, grade this essay following IELTS standards:

Essay: "{essay_to_grade}"

Provide your evaluation in this EXACT format in the begginning of your answer:
TR_Band: X.X
CC_Band: X.X
LR_Band: X.X
GRA_Band: X.X
Overall_Band: X.X
Explanation: [Brief justification for each criterion]
"""

    return prompt

TO BE DONE